# 05 · Abstract data visualization

*Notebooks: [01 Overview](01-overview.ipynb) · [02 Formats](02-formats.ipynb) · [03 Data](03-data.ipynb) · [04 Maps](04-maps.ipynb) · [05 Plots](05-plots.ipynb) · [06 Compute](06-compute.ipynb)*

`autk-plot` draws charts over the same features the map draws. That is its **feature-centric model**: a plot does not
get its own data, it reads the features of a table, so a mark in a chart *is* a feature on the map. Selecting marks
selects those features everywhere. In pyautark the plots are `Scatterplot`, `Histogram` and `Table`.

In [ ]:
import autark as ak
import tutorial as t

In [ ]:
neighborhoods = ak.GeoJSON("neighborhoods", url=t.url("data/map/mnt_neighs.geojson"))

## 1 A scatter plot

`ak.Scatterplot(source, x=..., y=...)` plots one mark per feature. `x` and `y` are property names; `color` names a
property to color by.

In [ ]:
ak.Spec(
    data=[neighborhoods],
    views=[ak.Scatterplot(neighborhoods, x="shape_area", y="shape_leng", name="area vs perimeter")],
).widget()

## 2 Histograms and `@transform`

A histogram bins one property. `ak.Histogram(source, x=..., bins=...)`. Behind the scenes the chart takes its bars from a
transform of the table (autk-plot's `binning-1d`), which is why its axis reads `@transform`.

In [ ]:
ak.Spec(
    data=[neighborhoods],
    views=[ak.Histogram(neighborhoods, x="shape_area", bins=12)],
).widget()

A `Table` lists properties and sorts them:

In [ ]:
ak.Spec(
    data=[neighborhoods],
    views=[ak.Table(neighborhoods, columns=["ntaname", "boroname", "shape_area"], sort={"column": "shape_area", "direction": "desc"})],
).widget()

## 3 Selections and links

### 3.1 Mouse events

A plot takes a `selection`: `ak.point(name)` selects one mark by click, `ak.interval(name)` selects a brushed region, and
histograms brush along x. The name identifies the selection in your code, and it is the key of the widget's `selections`
trait. Within the widget the interval selection is a rectangle you drag.

### 3.2 Linked visualizations

A `Link(selection, target=layer_id)` connects the selection to a map layer: selecting in the plot **highlights** those
features on the layer, and picking on the layer selects them in the plot. One spec can hold several plots and several
links. A grid layout puts them side by side; give a plot a `width` and `height` (pixels) so it fits its cell.

In [ ]:
brush = ak.interval("area_vs_perimeter")
bins = ak.interval("area_bins")

spec = ak.Spec(
    data=[neighborhoods],
    views=[
        ak.Map(name="map", layers=[ak.Layer(neighborhoods, id="neighborhoods").style(color="#9ecae9")]),
        ak.Scatterplot(neighborhoods, x="shape_area", y="shape_leng", selection=brush, width=460, height=380),
        ak.Histogram(neighborhoods, x="shape_area", bins=12, selection=bins, width=460, height=380),
    ],
    links=[ak.Link(brush, target="neighborhoods"), ak.Link(bins, target="neighborhoods")],
    layout=ak.Layout(type="grid", columns=2),
)
widget = spec.widget()
widget

Brush a region of the scatter plot or a range of bars: the same neighborhoods light up on the map.

### Selections in Python

The widget reports each plot's selection back to the kernel in `widget.selections`, as `{selection name: [feature ids]}`.
Select something above, then run this cell:

In [ ]:
widget.selections

`widget.observe(handler, names="selections")` calls a function on every change, so you can drive other Python code
from a brush, for instance to print the selected neighborhoods:

In [ ]:
features = t.read_json("data/map/mnt_neighs.geojson")["features"]

def show_selection(change):
    for name, ids in change["new"].items():
        print(name, "->", [features[i]["properties"]["ntaname"] for i in ids][:5], "..." if len(ids) > 5 else "")

widget.observe(show_selection, names="selections")

Brush again above; the cell prints the names under the selection (the output appears below the cell that registered
the handler).

### Limits

A link highlights; it does not filter. Each selection is tied to one plot, and a plot can highlight several layers
through several links.